In [ ]:
import os
import sys
import pickle
sys.path.append(os.path.abspath('..'))

import pickle
import numpy as np
from src.baselines import simulated_annealing, local_search
from src.qulacs_vqe import qubo_value
from src.metrics import calculate_budget, calculate_selected_actions, evaluate_robustness, calculate_optimality_gap
from src.plotting import plot_scenario_robustness_boxplot, plot_risk_vs_budget

ImportError: cannot import name 'simulated_annealing' from 'src.baselines' (/job/qulacs/Fujitsu-De-escalation-War/src/baselines.py)

In [ ]:
with open('results/raw/qubo_ising_data.pkl', 'rb') as f:
    data_n2 = pickle.load(f)
q, Q, const = data_n2['q'], data_n2['Q'], data_n2['const']

In [ ]:
with open('results/raw/vqe_results.pkl', 'rb') as f:
    data_n3 = pickle.load(f)
vqe_best_energy = data_n3.get('best_energy_hva', data_n3.get('best_energy'))
vqe_top_portfolios = data_n3.get('top_portfolios_hva', data_n3.get('top_portfolios'))
vqe_x = vqe_top_portfolios[0][0]  # Ambil bitstring terbaik

with open('results/raw/instance_data.pkl', 'rb') as f:
    data_n1 = pickle.load(f)
scenarios = data_n1['scenarios']
alpha = data_n1['alpha']

m = len(vqe_x)
costs = np.ones(m) 
print("Data QUBO, Scenarios, Alpha, dan hasil VQE berhasil di-load!")

In [ ]:
print("\nMenjalankan Simulated Annealing...")
sa_x, sa_fx = simulated_annealing(q, Q, const, n_steps=20000)

In [ ]:
print("Menjalankan Local Search dari bitstring acak...")
random_start = np.random.randint(0, 2, size=30)
ls_x = local_search(random_start, q, Q, max_sweeps=100)
ls_fx = qubo_value(ls_x, q, Q, const)

In [ ]:
print(f"VQE (Quantum Simulator) : {vqe_best_energy:.4f}")
print(f"Simulated Annealing     : {sa_fx:.4f}")
print(f"Local Search            : {ls_fx:.4f}")

In [ ]:
print("--- Analisis Komposisi Portofolio ---")
for name, x in zip(['VQE (HVA)', 'Simulated Annealing', 'Local Search'], [vqe_x, sa_x, ls_x]):
    budget_used = calculate_budget(x, costs)
    actions_selected = calculate_selected_actions(x)
    print(f"{name:20s} | Actions: {actions_selected:2d} | Budget: {budget_used:.2f}")

In [ ]:
print("--- Analisis Robustness (Ketahanan Skenario) ---")
vqe_robustness = evaluate_robustness(vqe_x, scenarios, alpha)
sa_robustness = evaluate_robustness(sa_x, scenarios, alpha)
ls_robustness = evaluate_robustness(ls_x, scenarios, alpha)

for name, rob in zip(['VQE (HVA)', 'Simulated Annealing', 'Local Search'], 
                     [vqe_robustness, sa_robustness, ls_robustness]):
    print(f"{name:20s} | Mean Risk: {rob['mean_risk']:.4f} | Worst Risk: {rob['worst_risk']:.4f}")

In [ ]:
print("--- Analisis Optimality Gap ---")
print("Asumsi: Hasil Simulated Annealing adalah near-optimum.")
vqe_gap = calculate_optimality_gap(vqe_best_energy, sa_fx)
ls_gap = calculate_optimality_gap(ls_fx, sa_fx)
print(f"Optimality Gap VQE : {vqe_gap:.2%}")
print(f"Optimality Gap LS  : {ls_gap:.2%}")

In [ ]:
# Plot Boxplot Robustness
print("Visualisasi Robustness (Distribusi Risiko residual antar Skenario):")
plot_scenario_robustness_boxplot(np.array(vqe_robustness['raw_risks']), np.array(sa_robustness['raw_risks']))

# Plot Risk vs Budget untuk ketiga portfolio
print("\nVisualisasi Risk vs Budget Utilized:")
portfolios_data = [
    {'budget': calculate_budget(vqe_x, costs), 'risk': vqe_robustness['mean_risk']},
    {'budget': calculate_budget(sa_x, costs), 'risk': sa_robustness['mean_risk']},
    {'budget': calculate_budget(ls_x, costs), 'risk': ls_robustness['mean_risk']}
]
plot_risk_vs_budget(portfolios_data)